In [ ]:
import os
import sys

sys.path.append("../")

envkey = "OMP_NUM_THREADS"
# Set this environment variable to the number of available cores in your machine,
# to get a fast execution of the Einstein Boltzmann Solver
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))
os.environ[envkey] = str(12)
print("The value of {:s} is: ".format(envkey), os.environ.get(envkey))

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
from copy import copy, deepcopy
import numpy as np
import seaborn
from getdist.gaussian_mixtures import GaussianND
from getdist import plots
from scipy.optimize import curve_fit
from scipy.interpolate import UnivariateSpline, RectBivariateSpline
from matplotlib import cm, colors
from matplotlib.ticker import LogLocator, FuncFormatter
import niceplots.utils as nicepl
from nautilus import Sampler, Prior
from scipy.integrate import quad

nicepl.initPlot()
Cs = seaborn.color_palette("colorblind")
Cp = seaborn.color_palette("Paired")
Cs

In [ ]:
# Import Main modules. This might take some time as some functions compile before time
from SSLimPy.interface import sslimpy
from SSLimPy.cosmology import cosmology
from SSLimPy.cosmology import halo_model
from SSLimPy.cosmology import astro
from SSLimPy.LIMsurvey import covariance as scov
from SSLimPy.LIMsurvey import power_spectrum as spobs

In [ ]:
settings = {
    "code":"class", # The Einstein--Boltzman solver that should be used
    "do_RSD" : False, # If RSD should be considerd
    "nonlinearRSD" : False, # If you want to add FOG to the RSD
    "QNLpowerspectrum": False, # Use dewiggled power spectrum (vlasov approximation of nonlinear structure formation)
    "FoG_damp" : "ISTF_like", # The particular parametrization for the FOG. Check PowerSpectrum for the full list
    "halo_model_PS" : True, # If the cosmological shotnoise should be computed from the halo model 
    "output" : ["Power spectrum", "Covariance"], # What output one wants (here power spectrum and Gaussian covariance only)
    "kmin": 1e-4 * u.Mpc**-1,
    "kmax": 50 * u.Mpc**-1,
    "nk": 200,
}

cosmodict={
    "h": 0.677,
    "Omegam": 0.309167,
    "Omegab": 0.04903,
    "sigma8":0.8222,
    "ns":0.96824,
    "mnu":0.06,
    "Neff":3.044,
}

# Parameters that enter your halo model. Typically they are not changed but you could
halodict={
    "halo_tracer" : "clustering", # Computes all halo quantities from the matter field - neutrinos
    "hmf_model": "ST", # Sheth--Tormann halo mass function
    "concentration": "Diemer19", # Diemer19 halo concentration relation
    "bias_model": "ST99",
}

# Parameters for the Survey specifications
z = np.array([1])
def get_specs(nu):
        nuObs = nu / (z + 1)
        DeltaDeltanu = 0.05143

        surveyspecs = {
                "Tsys_NEFD": 0 * u.uK, #System temperature for instrumental shotnoise
                "Nfeeds": 19,
                "tobs": 1300 * u.h,
                "nD": 1, # Observational parameters
                "beam_FWHM": 0. * u.arcmin,
                "nu":  nu,
                "nuObs": nuObs, # Observed Frequency
                "Delta_nu": DeltaDeltanu * nuObs, # Frequency Bin
                "dnu": 0. * u.MHz, # Spectrograph resolution
                "Omega_field": 18.64 * u.deg**2, # Angular size of survey
        }
        return surveyspecs

astrodict_CII={
    "model_type": "ML",
    "model_name": "SilvaCII",
    "model_par": {
        "a": 0.8475,
        "b": 7.2203,
        "SFR_file": "sfr_release.dat",
        "do_quench": False,
    },
    "sigma_scatter" : 0.37,
    "meanperserve_scatter": False,
}

nu = 1.897 * u.THz
surveyspecs_CII = get_specs(nu)

In [ ]:
myssl = sslimpy.SSLimPy(
    settings_dict=settings,
    cosmopars=cosmodict,
    halopars=halodict,
    astropars=astrodict_CII,
    obspars_dict=surveyspecs_CII,
)

myastro = myssl.current_astro
mycosmo = myastro.cosmology
h = mycosmo.h()

In [ ]:
response8 = np.load("/home/sefa/Desktop/projects/LIM-Code/clusterdata/CII_A_438681_Response_NSUB8.npz")
dTb = response8["deltab"]

k, Pk  = response8["k"], response8["Pkmean"]

#sanitize
mask = ~np.any(np.isnan(Pk), axis=0)
k = k[mask]
Pk = Pk[:, mask]

In [ ]:
mP, pP = np.percentile(Pk, [16, 84], axis=0)
plt.loglog(k, Pk.mean(0), color="royalblue")
plt.fill_between(k, mP, pP, color="royalblue", alpha=0.2)

In [ ]:
def linear(x, y):
    ym = y.mean(0)
    xm = x.mean(0)
    a = np.sum((y - ym) * (x - xm), axis=0) / np.sum((x- xm)**2, axis=0)
    b = ym - a * xm
    return a, b
    

dlogPddTb_arr = []
P0_arr = []
for pki in Pk.T:
    a, b = linear(dTb, np.log(pki))
    dlogPddTb_arr.append(a)
    P0_arr.append(b)

dlogPddTb_arr = np.array(dlogPddTb_arr)
P0_arr = np.array(P0_arr)

In [ ]:
Pk_clean = np.exp(np.log(Pk) - dlogPddTb_arr * dTb[:, None])
DPk_clean = Pk_clean - np.exp(P0_arr)
N, Nk = DPk_clean.shape

cov = np.einsum("ij, ik-> jk", DPk_clean, DPk_clean) / (N - 1)
corr = cov / np.sqrt(np.outer(np.diag(cov), np.diag(cov)))

In [ ]:
plt.imshow(corr, cmap="coolwarm", origin="lower", vmin=-1, vmax=1)
plt.colorbar()

In [ ]:
mP, pP = np.percentile(Pk, [16, 84], axis=0)
plt.loglog(k, Pk.mean(0), color="royalblue")
plt.fill_between(k, mP, pP, color="royalblue", alpha=0.2)

plt.errorbar(k, Pk_clean.mean(0), np.diag(np.sqrt(cov)),color="coral", ls="--")
plt.ylim(0.8, 3e2)

In [ ]:
def Weff(phi, mu, x):
    x1 = x * np.sqrt(1 - mu**2) * np.cos(phi)
    x2 = x * np.sqrt(1 - mu**2) * np.sin(phi)
    x3 = x * mu
    coords = np.array([x1, x2, x3])
    sincs = np.sinc(coords / 2 / np.pi)
    return np.prod(sincs**2)

def phi_integration(mu, x):
    result, _ = quad(Weff, -np.pi, np.pi, (mu, x))
    return result

def mu_integration(x):
    result, _ = quad(phi_integration, -1, 1, (x, ))
    return result

kth = k * myastro.Mpch**-1
deltax = 1 * myastro.Mpch
x = (deltax * kth).to(1).value
Weff_iso = np.vectorize(mu_integration)(x) / (4 * np.pi)

In [ ]:
Pk_th = (
    0.78 * myastro.Tbavg(1,1,1)**2 * mycosmo.matpow(kth, 1, False, "matter") +  1.1 *  myastro.Tbavg(0,1,2)
).to(u.uK**2 * myastro.Mpch**3)


In [ ]:
dTbdb_sims = 1.04 * 0.2284 * u.uK
dTbdb_sims

In [ ]:
dTbdb_th = myastro.Tbavg(1, 1, 1) * 0.81661479
dTbdb_th

In [ ]:
100 * (dTbdb_sims / dTbdb_th - 1)

In [ ]:
b11 = myastro.Tbavg(1,1,1)
b02 = myastro.Tbavg(0,1,2)
Pk_mm = mycosmo.matpow(kth, 1, False, "matter")


def Pk_th(pars):
    I11 = b11 * pars["b1"]
    sn = b02 * pars["sn"]
    eta = pars.get("eta", 1)    

    P2h = (4 * np.pi * (kth / (2 * np.pi))**3 * I11**2 * Pk_mm).to(u.uK**2) * f2d
    P1h = (4 * np.pi * (kth / (2 * np.pi))**3 * sn).to(u.uK**2)
    return (
        (4 * np.pi * (kth / (2 * np.pi))**3)**-1 * (P2h**eta + P1h**eta)**(1 / eta)
    ).to(myastro.Mpch**3 * u.uK**2).value

In [ ]:
Pk_data = Pk_clean.mean(0) / Weff_iso**2
cov_data = cov / np.outer(Weff_iso, Weff_iso)
invcov_data = np.linalg.inv(cov_data)

def likelihood(pars):
    Pk_model = Pk_th(pars)
    D = Pk_model - Pk_data
    return -0.5 * np.einsum("i, ij, j", D, invcov_data, D)

In [ ]:
prior = Prior()
prior.add_parameter("eta", (0.8, 1.2))
prior.add_parameter("b1", (0.9, 1.5))
prior.add_parameter("sn", (0.9, 1.8))

In [ ]:
sampler = Sampler(prior, likelihood, n_live=1000, pool=14)

In [ ]:
sampler.run(verbose=True)

In [ ]:
import corner
import matplotlib.pyplot as plt

points, log_w, log_l = sampler.posterior()
fig =corner.corner(
    points, weights=np.exp(log_w), bins=20, labels=prior.keys, color='coral',
    plot_datapoints=False, range=np.repeat(0.999, len(prior.keys)))


In [ ]:
popt = (points * np.exp(log_w)[:, None]).sum(0)
popt_dic = dict(eta=popt[0], b1=popt[1], sn=popt[2], logf=popt[3], nd=popt[4])
# popt_dic = dict(b1=popt[0], sn=popt[1])

In [ ]:
popt_dic

In [ ]:
plt.errorbar(k, Pk_data, np.sqrt(np.diag(cov_data)))
plt.loglog(k, Pk_th(popt_dic))